# T02 음성 명령어 인식 — 데이터 → 학습 → 평가 통합 노트북 (중간 점검)

Speech Commands v0.02 (35개 단어)로 0번 기준, 비딥러닝 기준, 기준선 A(사전 학습 wav2vec2), 팀 모델 B(로그 멜 CNN)를 한 노트북에서 차례로 학습하고 검증 묶음으로 비교합니다. 마지막 셀에 모든 실험의 결과 표가 나옵니다.

| 번호 | 실험 | 셀 |
|---|---|---|
| E01 | 0번 기준: 학습 묶음에서 가장 많은 단어만 고르기 | 2 |
| E05 | 비딥러닝 기준: MFCC 통계(평균·표준편차) + 로지스틱 회귀 | 5 |
| E02 | 기준선 A: wav2vec2-base 고정, 12층(마지막) 특징 + Linear | 6 |
| E03 | 기준선 A에서 특징 층만 바꿈 (1~12층 중 검증 최고) | 6 |
| E04 | 팀 모델 B: 로그 멜 스펙트로그램 CNN을 처음부터 학습 | 7 |

- Colab: 런타임 유형 T4 GPU → 모두 실행 (T4에서 약 10분: 내려받기, 특징 뽑기, 학습 포함)
- `SMOKE = True`: 단어마다 학습 20개·검증 5개, 1에폭으로 끝까지 도는지만 확인 (전체 실행 전에 한 번, GPU가 없을 때)
- 설정과 모델 선택은 검증 묶음으로만 합니다. 시험 묶음은 최종 평가 때 한 번만 쓰므로 이 노트북에서는 불러오지 않습니다
- 뽑은 특징과 스펙트로그램은 `cache/`에 저장해 다시 실행하면 건너뜁니다 (런타임이 끊기면 다시 계산)
- 라이브러리: 내려받기·분할·멜 스펙트로그램·MFCC는 torchaudio, wav 읽기는 soundfile (torchaudio 2.9부터 `torchaudio.load`가 torchcodec을 요구)

In [ ]:
# 셀 1. 설정과 버전
import os, sys, time, random, collections, copy
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt

SMOKE = False                  # True: 단어마다 학습 20개·검증 5개, 1에폭 (코드 확인용, 결과 숫자는 쓰지 않음)
SEED = 0
DATA_ROOT = "data"
CACHE_DIR = "cache"
SR = 16000
N_FFT, HOP, N_MELS = 400, 160, 64        # 25ms 창, 10ms 간격, 멜 밴드 64개
N_MFCC = 40                              # 비딥러닝 기준의 MFCC 개수
A_EPOCHS, A_LR = 30, 1e-3                # 기준선 A의 Linear
B_EPOCHS, B_LR, B_BATCH = 20, 1e-3, 256  # 팀 모델 B
B_CHANNELS = [16, 32, 64, 128]
LR_EPOCHS, LR_LR = 200, 1e-2             # 비딥러닝 기준의 로지스틱 회귀 (전체 배치)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = device.type == "cuda"              # T4에서 혼합 정밀도
if SMOKE:
    A_EPOCHS = B_EPOCHS = 1
    LR_EPOCHS = 5
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchaudio", torchaudio.__version__,
      "| soundfile", sf.__version__, "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "GPU 없음 → CPU", "| SMOKE", SMOKE)

RESULTS = []                             # 마지막 셀의 결과 표에 모음
def add_result(eid, setting, acc, f1, sec, note=""):
    RESULTS.append((eid, setting, acc, f1, sec, note))

In [ ]:
# 셀 2. 데이터, 공식 분할, 0번 기준 (E01)
os.makedirs(DATA_ROOT, exist_ok=True)
t0 = time.time()
splits = {s: torchaudio.datasets.SPEECHCOMMANDS(DATA_ROOT, download=True, subset=sub)
          for s, sub in [("train", "training"), ("val", "validation")]}
BASE_DIR = os.path.dirname(splits["train"]._path)
print(f"데이터 준비 {time.time() - t0:.0f}초 | {splits['train']._path}")

items = {s: [] for s in splits}                      # 분할마다 (경로, 단어, 화자)
for s, ds in splits.items():
    for i in range(len(ds)):
        path, _, word, spk, _ = ds.get_metadata(i)
        items[s].append((path, word, spk))
WORDS = sorted({w for _, w, _ in items["train"]})
W2I = {w: i for i, w in enumerate(WORDS)}
K = len(WORDS)
print(f"단어 {K}개 | 학습 {len(items['train']):,} / 검증 {len(items['val']):,} (공식 목록) | 시험 묶음은 불러오지 않음")
print("학습·검증 화자 겹침:", len({x[2] for x in items["train"]} & {x[2] for x in items["val"]}), "명")

count = collections.Counter(w for _, w, _ in items["train"])
print(f"학습 묶음 단어별: 최다 {count.most_common()[0]}, 최소 {count.most_common()[-1]} ({count.most_common()[0][1] / count.most_common()[-1][1]:.1f}배)")

if SMOKE:                                            # 단어마다 앞에서부터 몇 개만
    for s, k in [("train", 20), ("val", 5)]:
        per = collections.defaultdict(list)
        for it in items[s]:
            per[it[1]].append(it)
        items[s] = [it for w in WORDS for it in per[w][:k]]
    print("SMOKE:", {s: len(v) for s, v in items.items()})

def metrics(pred, y, k=K):
    """정확도, 매크로 F1, 혼동 행렬 (행 = 정답, 열 = 예측)"""
    pred, y = pred.cpu(), y.cpu()
    cm = torch.bincount(y * k + pred, minlength=k * k).reshape(k, k).float()
    tp = cm.diag()
    prec, rec = tp / cm.sum(0).clamp(min=1), tp / cm.sum(1).clamp(min=1)
    f1 = 2 * prec * rec / (prec + rec).clamp(min=1e-12)
    return (tp.sum() / cm.sum()).item(), f1.mean().item(), cm

# E01: 학습 묶음에서 가장 많은 단어를 검증 녹음 전부에 답함
count_now = collections.Counter(w for _, w, _ in items["train"])
majority = count_now.most_common(1)[0][0]
y_val = torch.tensor([W2I[w] for _, w, _ in items["val"]])
acc, f1, _ = metrics(torch.full_like(y_val, W2I[majority]), y_val)
add_result("E01", f"0번 기준: 가장 많은 단어({majority})만 고름", acc, f1, 0.0, "학습 없음")
print(f"E01 0번 기준({majority}): 검증 정확도 {acc:.2%}, 매크로 F1 {f1:.4f}")

In [ ]:
# 셀 3. 소리 한 번 읽기 → 로그 멜 스펙트로그램(팀 모델 B용)과 MFCC 통계(비딥러닝 기준용)를 함께 계산해 저장
mel = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS).to(device)
to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80).to(device)
mfcc = torchaudio.transforms.MFCC(sample_rate=SR, n_mfcc=N_MFCC,
                                  melkwargs={"n_fft": N_FFT, "hop_length": HOP, "n_mels": N_MELS}).to(device)

class Wavs(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, word, _ = self.items[i]
        x, _ = sf.read(os.path.join(BASE_DIR, path), dtype="float32")
        x = torch.from_numpy(x)
        return torch.nn.functional.pad(x, (0, max(0, SR - len(x))))[:SR], W2I[word]     # 1초로 맞춤

def loader(split, batch):
    return torch.utils.data.DataLoader(Wavs(items[split]), batch_size=batch, shuffle=False,
                                       num_workers=2 if os.name != "nt" else 0)

def cached(name, fn):
    path = os.path.join(CACHE_DIR, f"{name}{'_smoke' if SMOKE else ''}.pt")
    if os.path.exists(path):
        d = torch.load(path)
        print(f"  {name}: 저장된 것 사용 (처음 계산 {d['sec']:.0f}초)")
        return d
    d = fn()
    os.makedirs(CACHE_DIR, exist_ok=True)
    torch.save(d, path)
    return d

def spec_and_mfcc(split):
    def run():
        M, F, Y = [], [], []
        t0 = time.time()
        with torch.no_grad():
            for x, y in loader(split, 256):
                x = x.to(device)
                M.append(to_db(mel(x)).half().cpu())                     # (B, 64, 101)
                c = mfcc(x)                                              # (B, 40, 101)
                F.append(torch.cat([c.mean(-1), c.std(-1)], 1).cpu())    # 시간 평균·표준편차 → 80차원
                Y.append(y)
        d = {"mel": torch.cat(M).unsqueeze(1), "mfcc": torch.cat(F), "Y": torch.cat(Y), "sec": time.time() - t0}
        print(f"  {split}: 로그 멜 {tuple(d['mel'].shape)}, MFCC 통계 {tuple(d['mfcc'].shape)}를 {d['sec']:.0f}초에 계산")
        return d
    return cached(f"spec_{split}", run)

feats = {s: spec_and_mfcc(s) for s in ["train", "val"]}
SPEC_SEC = sum(d["sec"] for d in feats.values())

In [ ]:
# 셀 4. 예시 2개 — 파형과 로그 멜 스펙트로그램 (학습 묶음, 시드 고정)
rng = random.Random(SEED)
examples = [rng.choice([it for it in items["train"] if it[1] == w]) for w in ["yes", "three"]]
fig, axes = plt.subplots(2, 2, figsize=(12, 5.5))
for row, (path, w, spk) in enumerate(examples):
    x, _ = sf.read(os.path.join(BASE_DIR, path), dtype="float32")
    axes[row, 0].plot(np.arange(len(x)) / SR, x, lw=0.6); axes[row, 0].set_xlim(0, 1)
    axes[row, 0].set_title(f'"{w}" waveform ({len(x) / SR:.2f} s)'); axes[row, 0].set_xlabel("time (s)")
    xx = torch.nn.functional.pad(torch.from_numpy(x), (0, max(0, SR - len(x))))[:SR].to(device)
    with torch.no_grad():
        m = to_db(mel(xx)).cpu()
    axes[row, 1].imshow(m.numpy(), origin="lower", aspect="auto", extent=[0, 1, 0, N_MELS], cmap="magma")
    axes[row, 1].set_title(f'"{w}" log-mel spectrogram {tuple(m.shape)}'); axes[row, 1].set_xlabel("time (s)"); axes[row, 1].set_ylabel("mel band")
plt.tight_layout(); plt.show()

In [ ]:
# 셀 5. E05 비딥러닝 기준 — MFCC 통계(80차원) + 다항 로지스틱 회귀 (사람이 정한 특징 + 선형 분류기)
def fit_linear(Xtr, Ytr, Xva, Yva, epochs, lr, batch=None, seed=SEED):
    """학습 묶음 통계로 표준화한 특징 위에 Linear 하나(= 로지스틱 회귀)를 학습. 검증 정확도 최고 에폭의 결과"""
    torch.manual_seed(seed)
    Xtr, Ytr, Xva, Yva = Xtr.float().to(device), Ytr.to(device), Xva.float().to(device), Yva.to(device)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xtr, Xva = (Xtr - mu) / sd, (Xva - mu) / sd
    head = nn.Linear(Xtr.shape[1], K).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    best, t0 = None, time.time()
    for ep in range(epochs):
        head.train()
        perm = torch.randperm(len(Xtr), device=device)
        step = batch or len(Xtr)
        for i in range(0, len(Xtr), step):
            idx = perm[i:i + step]
            loss = nn.functional.cross_entropy(head(Xtr[idx]), Ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            acc, f1, cm = metrics(head(Xva).argmax(1), Yva)
        if best is None or acc > best["acc"]:
            best = {"acc": acc, "f1": f1, "epoch": ep + 1, "cm": cm}
    best["sec"] = time.time() - t0
    best["params"] = sum(p.numel() for p in head.parameters())
    return best

E05 = fit_linear(feats["train"]["mfcc"], feats["train"]["Y"], feats["val"]["mfcc"], feats["val"]["Y"], LR_EPOCHS, LR_LR)
add_result("E05", f"비딥러닝 기준: MFCC {N_MFCC}개 평균·표준편차 + 로지스틱 회귀", E05["acc"], E05["f1"], E05["sec"],
           f"파라미터 {E05['params']:,}")
print(f"E05 비딥러닝 기준: 검증 정확도 {E05['acc']:.2%}, 매크로 F1 {E05['f1']:.4f} (최고 에폭 {E05['epoch']}/{LR_EPOCHS}, {E05['sec']:.1f}초)")

In [ ]:
# 셀 6. 기준선 A — 사전 학습 wav2vec2-base 고정, 층별 특징(시간 평균 768차원) + Linear (E02, E03)
bundle = torchaudio.pipelines.WAV2VEC2_BASE
w2v = bundle.get_model().to(device).eval()
for p in w2v.parameters():
    p.requires_grad = False
N_W2V = sum(p.numel() for p in w2v.parameters())
print(f"wav2vec2-base 파라미터 {N_W2V:,}개 (고정)")

def w2v_feats(split):
    def run():
        X, Y = [], []
        t0 = time.time()
        with torch.inference_mode(), torch.autocast(device.type, dtype=torch.float16, enabled=AMP):
            for x, y in loader(split, 64):
                layers, _ = w2v.extract_features(x.to(device))
                X.append(torch.stack([h.mean(dim=1) for h in layers], dim=1).half().cpu())   # (B, 12, 768)
                Y.append(y)
        d = {"X": torch.cat(X), "Y": torch.cat(Y), "sec": time.time() - t0}
        print(f"  {split}: {tuple(d['X'].shape)}를 {d['sec']:.0f}초에 뽑음")
        return d
    return cached(f"w2v_base_{split}", run)

wf = {s: w2v_feats(s) for s in ["train", "val"]}
W2V_SEC = sum(d["sec"] for d in wf.values())
layer_res = []
for L in range(12):
    r = fit_linear(wf["train"]["X"][:, L], wf["train"]["Y"], wf["val"]["X"][:, L], wf["val"]["Y"], A_EPOCHS, A_LR, batch=512)
    layer_res.append(r)
    print(f"  {L + 1:2d}층: 검증 정확도 {r['acc']:.2%}, 매크로 F1 {r['f1']:.4f} ({r['sec']:.1f}초)")
E02 = layer_res[11]
best_L = int(np.argmax([r["acc"] for r in layer_res]))
E03 = layer_res[best_L]
add_result("E02", "기준선 A: wav2vec2-base 12층(마지막) 고정 + Linear", E02["acc"], E02["f1"], W2V_SEC + E02["sec"],
           f"Linear {E02['params']:,}개 학습 / wav2vec2 {N_W2V:,}개 고정")
add_result("E03", f"E02에서 특징 층만 바꿈: {best_L + 1}층 (검증 최고)", E03["acc"], E03["f1"], E03["sec"], "특징은 E02와 같이 씀")

plt.figure(figsize=(7, 3))
plt.plot(range(1, 13), [r["acc"] for r in layer_res], "o-", label="val acc")
plt.plot(range(1, 13), [r["f1"] for r in layer_res], "s--", label="val macro F1")
plt.xticks(range(1, 13)); plt.xlabel("wav2vec2 transformer layer"); plt.ylim(0, 1); plt.grid(alpha=.3); plt.legend()
plt.title("Baseline A: frozen wav2vec2 layer + Linear"); plt.tight_layout(); plt.show()
del w2v
if device.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# 셀 7. 팀 모델 B — 로그 멜 스펙트로그램 CNN, 처음부터 학습 (E04)
class KWSCNN(nn.Module):
    def __init__(self, channels=B_CHANNELS, n_classes=K):
        super().__init__()
        blocks, c_in = [], 1
        for c in channels:
            blocks += [nn.Conv2d(c_in, c, 3, padding=1), nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c_in, n_classes)

    def forward(self, x):
        return self.fc(self.pool(self.features(x)).flatten(1))

MU, SD = feats["train"]["mel"].float().mean().item(), feats["train"]["mel"].float().std().item()
Xtr = ((feats["train"]["mel"].float() - MU) / SD).to(device); Ytr = feats["train"]["Y"].to(device)
Xva = ((feats["val"]["mel"].float() - MU) / SD).to(device);   Yva = feats["val"]["Y"].to(device)

def train_cnn(epochs=B_EPOCHS, lr=B_LR, seed=SEED, log=True):
    torch.manual_seed(seed)
    m = KWSCNN().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    scaler = torch.amp.GradScaler(enabled=AMP)
    hist, best, t_start = [], None, time.time()
    for ep in range(epochs):
        m.train()
        t0 = time.time(); tot = 0.0
        perm = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(Xtr), B_BATCH):
            idx = perm[i:i + B_BATCH]
            with torch.autocast(device.type, dtype=torch.float16, enabled=AMP):
                loss = nn.functional.cross_entropy(m(Xtr[idx]), Ytr[idx])
            opt.zero_grad()
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(idx)
        if device.type == "cuda":
            torch.cuda.synchronize()
        sec = time.time() - t0
        m.eval()
        with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=AMP):
            pred = torch.cat([m(Xva[i:i + 1024]).argmax(1) for i in range(0, len(Xva), 1024)])
        acc, f1, cm = metrics(pred, Yva)
        hist.append((tot / len(Xtr), acc, f1, sec))
        if best is None or acc > best["acc"]:
            best = {"acc": acc, "f1": f1, "epoch": ep + 1, "cm": cm, "state": copy.deepcopy(m.state_dict())}
        if log:
            print(f"  epoch {ep + 1:2d}  train loss {tot / len(Xtr):.4f} | val acc {acc:.4f} | val macro F1 {f1:.4f} | {sec:.1f}초")
    best["sec"] = time.time() - t_start
    best["hist"] = hist
    best["params"] = sum(p.numel() for p in m.parameters())
    return best

E04 = train_cnn()
add_result("E04", f"팀 모델 B: 로그 멜 CNN {B_CHANNELS}, 처음부터 학습", E04["acc"], E04["f1"], E04["sec"],
           f"파라미터 {E04['params']:,} ({E04['params'] / N_W2V:.2%} of wav2vec2), 에폭당 {np.mean([h[3] for h in E04['hist']]):.1f}초")
print(f"E04 팀 모델 B: 검증 정확도 {E04['acc']:.2%}, 매크로 F1 {E04['f1']:.4f} (최고 에폭 {E04['epoch']}/{B_EPOCHS}, 학습 {E04['sec']:.0f}초)")
os.makedirs(CACHE_DIR, exist_ok=True)
torch.save({"state": E04["state"], "mu": MU, "sd": SD, "words": WORDS}, os.path.join(CACHE_DIR, f"E04_cnn{'_smoke' if SMOKE else ''}.pt"))

h = np.array(E04["hist"])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(range(1, len(h) + 1), h[:, 0], "o-"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("train loss"); ax[0].grid(alpha=.3)
ax[1].plot(range(1, len(h) + 1), h[:, 1], "o-", label="val acc"); ax[1].plot(range(1, len(h) + 1), h[:, 2], "s--", label="val macro F1")
ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.suptitle("E04 team model B: log-mel CNN from scratch"); plt.tight_layout(); plt.show()

In [ ]:
# 셀 8. 오류 분석 준비 — 모델마다 가장 많이 헷갈린 단어 쌍 5개
for name, r in [("E05 비딥러닝", E05), ("E03 기준선 A(최고 층)", E03), ("E04 팀 모델 B", E04)]:
    cm = r["cm"].clone(); cm.fill_diagonal_(0)
    top = torch.topk(cm.flatten(), 5)
    pairs = [f"{WORDS[i // K]}→{WORDS[i % K]} {int(v)}" for v, i in zip(top.values.tolist(), top.indices.tolist())]
    print(f"{name:<22s} " + ", ".join(pairs))

In [ ]:
# 셀 9. 모든 실험의 결과 표 (검증 묶음)
order = {"E01": 0, "E05": 1, "E02": 2, "E03": 3, "E04": 4}
rows = sorted(RESULTS, key=lambda r: order.get(r[0], 99))
print(f"{'번호':<4} | {'바꾼 설정':<48} | {'검증 정확도':>8} | {'매크로 F1':>8} | {'시간(초)':>8} | 비고")
for eid, setting, acc, f1, sec, note in rows:
    print(f"{eid:<4} | {setting:<48} | {acc:>10.4f} | {f1:>9.4f} | {sec:>9.1f} | {note}")
print("SMOKE 모드 결과 (숫자를 쓰지 말 것)" if SMOKE else f"device {device} | 시드 {SEED} | 스펙트로그램·MFCC 계산 {SPEC_SEC:.0f}초, wav2vec2 특징 {W2V_SEC:.0f}초 (한 번만)")

plt.figure(figsize=(8, 3))
plt.barh([f"{r[0]}" for r in rows][::-1], [r[2] for r in rows][::-1], color="#4C72B0")
for i, r in enumerate(rows[::-1]):
    plt.text(r[2] + 0.01, i, f"{r[2]:.2%}", va="center")
plt.xlim(0, 1.1); plt.xlabel("validation accuracy"); plt.title("All experiments (validation)"); plt.tight_layout(); plt.show()